# Clase 12. Casos de uso reales: de los juegos clásicos a AlphaStar

**DIH DATalife — Curso de Reinforcement Learning (RL) desde cero**

Bienvenida o bienvenido a la última clase del curso. Llevas 11 clases construyendo un edificio: empezaste con el dilema de explorar y explotar en los bandidos, seguiste con los procesos de decisión de Markov (PDM) y la programación dinámica, aprendiste a predecir y controlar sin modelo con Monte Carlo, Diferencia Temporal (TD), SARSA y Q-learning, diste el salto a las redes neuronales con las redes Q profundas (DQN, por sus siglas en inglés *Deep Q-Network*) y los gradientes de política, y cerraste con planificación, Búsqueda en Árbol de Monte Carlo (MCTS, por sus siglas en inglés *Monte Carlo Tree Search*) y arquitecturas de agentes distribuidas. Todo eso, hasta ahora, ha sido en el fondo teoría de pizarra: fórmulas, pseudocódigo, entornos de juguete.

Esta clase final te enseña que esa teoría no se quedó en la pizarra. Vas a recorrer una línea de sistemas reales, de Deep Blue a AlphaStar, y en cada uno vas a reconocer piezas que ya conoces: una función de valor aquí, una política aprendida por gradiente allá, una búsqueda tipo MCTS más adelante. No memorices nombres ni fechas: fíjate en cómo las mismas ideas simples, combinadas y escaladas, bastaron para vencer a los mejores jugadores humanos del planeta en juegos que llevaban décadas —o siglos— considerándose baluartes del intelecto humano.

## Guion de la clase

1. De Deep Blue al Monte Carlo Tree Search (MCTS)
2. AlphaGo: la jugada que sorprendió al mundo
3. AlphaZero: aprender desde cero, sin datos humanos
4. AlphaStar: el reto de StarCraft II
5. Código en vivo: mini-AlphaZero para Tres en Raya, celda a celda
6. Puntos clave y cierre del curso

**Acrónimos que vas a encontrar hoy** (nombre completo la primera vez que aparecen, y aquí en orden alfabético para consulta rápida): APM (acciones por minuto), DQN (*Deep Q-Network*, redes Q profundas), GPI (Iteración de Política Generalizada), IA (inteligencia artificial), LSTM (*Long Short-Term Memory*, memoria de largo plazo), MCTS (*Monte Carlo Tree Search*), PDM (procesos de decisión de Markov), PPO (Optimización de Política Próxima), PUCT (*Predictor + UCT*), RL (*Reinforcement Learning*, aprendizaje por refuerzo), TD (Diferencia Temporal), TPU (unidad de procesamiento tensorial), UCT (*Upper Confidence bound for Trees*).


## 1. De Deep Blue al Monte Carlo Tree Search

*Guion de clase: arranca preguntando quién ha oído hablar de Deep Blue o de AlphaGo. Casi siempre alguien conoce el nombre pero no la mecánica interna; hoy conectas esa mecánica con lo que ya sabes.*

Antes de que ninguna red neuronal jugara al Go, la inteligencia artificial (IA) ya llevaba 30 años derrotando campeones humanos en juegos de mesa, con técnicas mucho más cercanas a la búsqueda que al aprendizaje. Los juegos funcionan como "moscas de la fruta" para la investigación en IA: reglas claras, objetivos medibles, un marcador que no miente, igual que la genética usó la Drosophila para descubrir principios que luego se generalizaron a organismos mucho más complejos.

**Chinook**, un programa de damas, fue el primer hito: alcanzó una partida prácticamente perfecta combinando poda alfa-beta con análisis retrógrado (resolver los finales de partida hacia atrás, desde posiciones de victoria ya conocidas). Su rival fue Marion Tinsley, probablemente el mejor jugador de damas de la historia: perdió solo cinco partidas en toda su carrera, y dos de ellas fueron contra Chinook. Después, Schaeffer y su equipo resolvieron el juego por completo.

En 1997 llegó **Deep Blue**, de IBM, que venció al campeón mundial de ajedrez Garri Kaspárov en un enfrentamiento que se convirtió en uno de los eventos más seguidos de internet en su momento. Deep Blue no aprendía nada: evaluaba unos 8.000 rasgos codificados a mano (seguridad del rey, estructura de peones, material) y examinaba hasta 200 millones de posiciones por segundo, mirando entre 16 y 40 jugadas por delante. Es, en esencia, minimax con poda alfa-beta: fuerza bruta guiada por una función de evaluación diseñada por humanos, sin rastro de aprendizaje por refuerzo (RL, por sus siglas en inglés *Reinforcement Learning*).

El contrapunto llegó con **TD-Gammon**, un programa de backgammon que aprendió una función de valor —la probabilidad de ganar desde una posición dada— entrenando una red neuronal pequeña mediante TD, únicamente jugando contra sí misma, sin ningún dato humano. Es la misma idea que ya conoces de predicción sin modelo con TD y de aproximar funciones de valor con una red neuronal, aplicada casi una década antes de que existiera el término "aprendizaje profundo". Dato curioso: al backgammon se juega con dados, y esa aleatoriedad ya proporcionaba exploración natural, así que TD-Gammon no necesitó una estrategia explícita de exploración como la ε-greedy de los bandidos.

Poco después, **Maven** venció al campeón mundial de Scrabble, Adam Logan, por nueve partidas a cinco, combinando una búsqueda Monte Carlo truncada (simular solo unos diez movimientos por delante, no partidas completas) con una función de evaluación aprendida. Esa búsqueda Monte Carlo truncada es el eslabón que conecta todo lo anterior con el **MCTS**: en vez de resolver el juego entero, simulas muchas continuaciones posibles desde la posición actual y aprendes una política local, específica de esa situación. Reconoces el ciclo de selección, expansión, simulación y retropropagación de la clase de planificación: aquí lo vuelves a encontrar como el puente exacto hacia AlphaGo.


## 2. AlphaGo: la jugada que sorprendió al mundo

*Guion de clase: antes de explicar la arquitectura, pregunta por qué la fuerza bruta de Deep Blue no serviría para el Go. Deja que la clase llegue sola al tamaño del árbol de búsqueda.*

El Go llevaba 3.000 años jugándose y tenía alrededor de 40 millones de jugadoras y jugadores activos cuando DeepMind decidió abordarlo. El reto no era solo su tablero de 19 por 19: cada posición admite unas 200 jugadas legales de media (frente a las 20-35 del ajedrez), y el número de configuraciones posibles supera al de átomos en el universo observable. La fuerza bruta de Deep Blue, aquí, simplemente no era una opción; el Go se juega más por intuición que por cálculo exhaustivo, y AlphaGo necesitaba aproximar esa intuición.

La arquitectura combinaba dos redes neuronales convolucionales: una **red de política**, que propone una distribución de probabilidad sobre las jugadas legales, y una **red de valor**, que estima un único número —la probabilidad de ganar— desde una posición dada. Reconoce ambas piezas: la red de valor es, en esencia, la misma función de valor que ya conoces, aprendida con una red neuronal; la red de política es un gradiente de política, entrenado primero por imitación y después por autojuego. Las dos redes se combinaban con MCTS mediante una variante llamada PUCT (*Predictor + UCT*, donde UCT es *Upper Confidence bound for Trees*): la política reduce la anchura de la búsqueda (qué jugadas merece la pena explorar) y la red de valor reduce su profundidad (evita jugar cada rama hasta el final). Con eso, AlphaGo llegaba a mirar entre 50 y 60 jugadas por delante.

El entrenamiento tuvo dos fases: primero, aprendizaje supervisado a partir de partidas de jugadoras y jugadores humanos expertos, para que la red de política imitara jugadas fuertes; después, refinamiento por autojuego, jugando millones de partidas contra versiones anteriores de sí misma, mientras la red de valor aprendía a predecir el resultado final de cada posición alcanzada. En unas 72 horas de entrenamiento, AlphaGo pasó de jugar como una persona principiante a un nivel sobrehumano.

Los resultados hablan solos: AlphaGo venció a la campeona europea Fan Hui por cinco partidas a cero, y en marzo de 2016, en Seúl, venció al legendario Lee Sedol —9-dan, 18 títulos mundiales— por cuatro partidas a una. La jugada más recordada fue la **jugada 37** de la segunda partida: un golpe de hombro en la quinta línea tan inusual que la propia AlphaGo calculó solo una probabilidad entre 10.000 de que un humano la jugara, y terminó siendo una genialidad estratégica. Pero el sistema también tuvo un límite: en la partida cuatro, Lee Sedol encontró la **jugada 78**, a la que AlphaGo había asignado apenas un 0,007% de probabilidad, y ganó esa partida. Fue un recordatorio útil, y honesto, de que "sobrehumano" no significa infalible. Meses después, AlphaGo Zero —entrenada desde cero, sin ningún dato humano— venció al 90% de las partidas a AlphaGo Master, la versión que a su vez había acumulado 60 victorias y ninguna derrota contra las mejores profesionales del mundo.


## 3. AlphaZero: aprender desde cero, sin datos humanos

*Guion de clase: escribe en la pizarra la fórmula de mejora de política y pregunta qué papel juega cada símbolo antes de seguir leyendo. Es el mismo GPI de la clase de control sin modelo, disfrazado.*

AlphaZero simplifica y generaliza todo lo anterior de un plumazo: una única red neuronal, con salida conjunta de política y valor, y un único algoritmo aplicado sin cambios a ajedrez, shōgi (el ajedrez japonés) y Go, cambiando solo la codificación del tablero. Nada de datos humanos, nada de aperturas ni finales precargados: empieza con pesos aleatorios y las reglas del juego, y mejora exclusivamente jugando contra sí misma.

El mecanismo que cierra el bucle es puro GPI —la Iteración de Política Generalizada, la misma alternancia de evaluar y mejorar que ya viste con SARSA y Q-learning—, solo que aquí MCTS hace el papel de "operador de mejora de política": la búsqueda produce una distribución de jugadas mejor que la que sugiere la red por sí sola, y la red se entrena después para imitar esa distribución mejorada. Formalmente:

π′(a|s) ∝ Q(s,a)^(1/τ)

donde π′ es la política mejorada, Q(s,a) es el valor estimado por MCTS para cada jugada y τ (tau) es un parámetro de temperatura: valores altos exploran más, valores bajos concentran la elección en la jugada más prometedora. La red se entrena con una pérdida de entropía cruzada para la parte de política y error cuadrático medio para la parte de valor, todo en una sola red.

Los resultados, con hardware especializado (unidades de procesamiento tensorial, TPU), llegaron con una velocidad que sorprendió incluso a sus creadoras y creadores: tras solo unas cuatro horas de autojuego, AlphaZero ya superaba a Stockfish, el motor de ajedrez más fuerte del momento; tras unas dos horas, superaba a Elmo, el mejor programa de shōgi; y tras unas ocho horas, superaba a la propia versión de AlphaGo que había vencido a Lee Sedol. Frente a Stockfish —que evalúa millones de posiciones por segundo con reglas de material fijadas a mano (peón = 1, caballo = alfil = 3, torre = 5, dama = 9)— AlphaZero desarrolló un estilo propio: sacrificar material por actividad de las piezas y control a largo plazo. En una partida documentada con apertura Ruy López, AlphaZero sacrificó la dama en la jugada 25 para desmontar la estructura de Stockfish y acabó ganando por jaque mate; en otra, con apertura siciliana, sacrificó una torre por un peón en la jugada 24 para asegurar una estructura de peones superior. Ningún manual de ajedrez le enseñó a hacer eso: lo descubrió jugando contra sí misma, del mismo modo en que Q-learning aprende una política óptima sin que nadie le diga de antemano cuál es.


## 4. AlphaStar: el reto de StarCraft II

*Guion de clase: pide a la clase que enumere qué hace a StarCraft II distinto del ajedrez o el Go antes de leer la respuesta. Suele salir "no se ve todo el mapa" y "hay que decidir en tiempo real", que es justo el punto.*

Si el Go y el ajedrez son juegos de información perfecta y turnos discretos, StarCraft II es justo lo contrario: información parcial (la famosa "niebla de guerra" oculta buena parte del mapa), millones de acciones posibles en cada instante y decisiones que hay que tomar en tiempo real, no por turnos, combinando planificación estratégica a largo plazo con microgestión de decenas de unidades a la vez. DeepMind construyó **AlphaStar** para jugar la raza Protoss en este entorno, en colaboración con Blizzard Entertainment.

El entrenamiento combinó dos fases que ya reconoces: primero, unos tres días de aprendizaje por imitación sobre partidas humanas reales, que llevaron al agente hasta un nivel aproximado de jugador "platino"; después, cerca de una semana de autojuego organizado en la llamada **Liga AlphaStar** —agentes principales, "explotadores" especializados en encontrar sus debilidades y agentes de prueba, compitiendo entre sí a un ritmo equivalente a cientos de años de partidas humanas—. Ese esquema de liga con múltiples agentes jugando en paralelo es la misma filosofía distribuida de los métodos actor-crítico distribuidos que ya viste, llevada a una escala mucho mayor; y el algoritmo de RL que ajustaba la política en cada partida fue la Optimización de Política Próxima (PPO, por sus siglas en inglés *Proximal Policy Optimization*), un método de gradiente de política que limita cuánto puede cambiar la política en cada actualización, para no desestabilizar el aprendizaje.

Arquitectónicamente, AlphaStar combinó redes convolucionales para procesar el mapa, memoria de largo plazo (LSTM, *Long Short-Term Memory*) para recordar información pasada bajo observación parcial, y mecanismos de atención para priorizar qué unidades importan en cada momento. Para que la comparación con humanos fuera justa, el equipo limitó su tiempo de reacción a unos 350 milisegundos y sus acciones por minuto (APM) a niveles comparables a los de una jugadora o jugador profesional, de modo que una victoria reflejara mejores decisiones y no velocidad sobrehumana de clic. El resultado: una victoria por diez a cero contra el profesional Grzegorz "MaNa" Komincz, tras haberse medido también contra Liquid TLO.

**Fíjate en el patrón que se repite en los cuatro sistemas**: exploración/explotación a escala de liga, autojuego que genera sus propios datos y gradientes de política entrenados con PPO para digerir todo eso, esta vez bajo información incompleta. Nada de esto es magia nueva: son los mismos ingredientes de siempre, combinados con más cómputo y más paciencia.


## 5. Código en vivo: mini-AlphaZero para Tres en Raya

*Guion de clase: este es el bloque central de la clase. Construye el código celda a celda, sin adelantar la celda del bucle principal. Antes de ejecutar cada demo de autojuego, proyecta la pantalla y pide a la clase que prediga el siguiente movimiento.*

Vas a implementar, a escala de juguete, el mismo patrón que acabas de ver en AlphaGo, AlphaZero y AlphaStar: autojuego, búsqueda tipo MCTS y mejora iterativa de la política. Juegas al Tres en Raya en vez de al Go, y usas una función heurística sencilla más una tabla de valores en lugar de una red neuronal entrenada de verdad, para que se pueda ejecutar en segundos en cualquier Google Colab, sin más dependencia que `numpy`.

**Aviso didáctico**: esto es una versión de juguete del patrón de AlphaZero. No es AlphaZero real: no hay red neuronal convolucional entrenada por descenso de gradiente, el "valor" se guarda en un diccionario en vez de una red, y el juego es Tres en Raya, no ajedrez ni Go. El objetivo es que veas el mismo engranaje —autojuego, búsqueda, mejora de política— funcionar de principio a fin en un juego trivial, no reproducir el sistema real.

**Ejecuta cada celda por orden.** Vas a encontrar, cada pocas celdas, una partida de autojuego impresa turno a turno: proyéctala y coméntala con la clase antes de seguir.


In [ ]:
import math
import random

import numpy as np

np.random.seed(0)
random.seed(0)


### El tablero

El tablero es una tupla de nueve casillas: `0` vacía, `1` una ficha de X, `-1` una ficha de O. `LINEAS_GANADORAS` enumera las ocho combinaciones de tres casillas que ganan la partida.


In [ ]:
LINEAS_GANADORAS = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6),
]


def tablero_nuevo():
    return tuple([0] * 9)  # 0 = vacio, 1 = X, -1 = O


def movimientos_validos(tablero):
    return [i for i, v in enumerate(tablero) if v == 0]


def aplicar_movimiento(tablero, pos, jugador):
    nuevo = list(tablero)
    nuevo[pos] = jugador
    return tuple(nuevo)


### El resultado de una partida

`resultado` recorre las ocho líneas ganadoras y comprueba si alguna suma +3 (gana X) o -3 (gana O). Si el tablero está lleno y nadie ganó, es empate. Si nada de eso ocurre, la partida sigue.


In [ ]:
def resultado(tablero):
    """Devuelve +1 si gana X, -1 si gana O, 0 si hay empate, None si sigue."""
    for a, b, c in LINEAS_GANADORAS:
        suma = tablero[a] + tablero[b] + tablero[c]
        if suma == 3:
            return 1
        if suma == -3:
            return -1
    if 0 not in tablero:
        return 0
    return None


### Una ayuda solo para clase: imprimir el tablero

Esta función no forma parte del algoritmo: la añadimos únicamente para poder proyectar partidas de autojuego durante la clase, casilla a casilla.


In [ ]:
def imprimir_tablero(tablero):
    """Imprime el tablero en tres filas, con X, O y punto para las casillas vacias.
    Solo para visualizar en clase: el algoritmo nunca llama a esta funcion."""
    simbolos = {1: "X", -1: "O", 0: "."}
    for f in range(3):
        fila = tablero[f * 3:(f + 1) * 3]
        print(" ".join(simbolos[v] for v in fila))
    print()


imprimir_tablero(tablero_nuevo())


### La heurística y la tabla de valores

`heuristica` es la sustituta mínima de la red de valor de AlphaZero: para cada línea de tres casillas, si solo hay piezas de X (o ninguna) suma piezas propias al cuadrado; si solo hay piezas de O, resta lo mismo. Es una función fija, no aprendida: solo sirve como estimación de partida cuando la tabla de valores todavía no ha visto una posición. Devuelve un valor aproximado en [-1, 1] desde la perspectiva de X.

`tabla_valores` es un diccionario, `estado -> valor estimado`, que cumple, a escala mínima, el papel que en AlphaZero cumple la red de valor entrenada: la vas a rellenar con autojuego dentro de un momento.


In [ ]:
def heuristica(tablero):
    """Sustituta minima de la red de valor de AlphaZero. Devuelve un valor
    aproximado en [-1, 1] desde la perspectiva de X."""
    puntuacion = 0
    for linea in LINEAS_GANADORAS:
        valores = [tablero[i] for i in linea]
        piezas_x = valores.count(1)
        piezas_o = valores.count(-1)
        if piezas_o == 0:
            puntuacion += piezas_x ** 2
        if piezas_x == 0:
            puntuacion -= piezas_o ** 2
    return math.tanh(puntuacion / 6.0)


# Tabla de valores aprendida por autojuego: estado -> valor estimado
# desde la perspectiva de X (+1 = X gana, -1 = O gana, 0 = tablas).
tabla_valores = {}


def valor_estimado(tablero):
    if tablero in tabla_valores:
        return tabla_valores[tablero]
    return heuristica(tablero)


### El nodo del árbol de búsqueda

Cada `Nodo` guarda su tablero, de quién es el turno, sus hijos ya explorados, cuántas veces se ha visitado (`N`) y la suma acumulada de valores que ha recibido (`W`). `valor_medio` es simplemente `W / N`: el promedio de todas las simulaciones que pasaron por este nodo.


In [ ]:
class Nodo:
    __slots__ = ("tablero", "jugador", "hijos", "N", "W", "expandido")

    def __init__(self, tablero, jugador):
        self.tablero = tablero
        self.jugador = jugador  # jugador (+1 o -1) que debe mover aqui
        self.hijos = {}
        self.N = 0
        self.W = 0.0
        self.expandido = False

    def valor_medio(self):
        return self.W / self.N if self.N > 0 else 0.0


### UCT: cómo decidir qué rama explorar

`ucb_score` combina "cuánto de bueno parece" un hijo (`q`, convertido a la perspectiva del jugador que mueve en el padre) con "cuánto se ha explorado" (`u`, que crece si el hijo se ha visitado poco). Es la misma lógica de Upper Confidence Bound (UCB) que ya usaste con los bandidos, aplicada ahora a un árbol: por eso se llama UCT (*Upper Confidence bound for Trees*). En AlphaZero, `u` también incorpora la probabilidad previa de la red de política (eso es justo lo que añade PUCT); aquí usamos exploración uniforme para mantener el ejemplo simple.


In [ ]:
def ucb_score(padre, hijo, c=1.4):
    q = hijo.valor_medio() * padre.jugador
    u = c * math.sqrt(math.log(padre.N + 1) / (hijo.N + 1))
    return q + u


### El ciclo de MCTS: selección, expansión, evaluación y retropropagación

`simular` es el corazón del ejemplo. Cada llamada baja por el árbol hasta encontrar un nodo sin expandir, lo expande creando un hijo por cada movimiento válido, evalúa esa posición nueva y propaga el valor hacia arriba actualizando `N` y `W` de cada nodo del camino.

Fíjate en un detalle importante: en vez de un *rollout* aleatorio hasta el final de la partida (el MCTS "clásico" que viste en la clase de planificación), evaluamos la posición directamente con `valor_estimado` —la tabla de valores o, si aún no la conoce, la heurística—. Es exactamente lo que hace AlphaZero: sustituye el *rollout* por su red de valor.


In [ ]:
def simular(nodo):
    """Un ciclo de seleccion, expansion, evaluacion y retropropagacion."""
    r = resultado(nodo.tablero)
    if r is not None:
        nodo.N += 1
        nodo.W += r
        return r

    if not nodo.expandido:
        # --- Expansion: un hijo por cada movimiento valido ---
        for m in movimientos_validos(nodo.tablero):
            tablero_hijo = aplicar_movimiento(nodo.tablero, m, nodo.jugador)
            nodo.hijos[m] = Nodo(tablero_hijo, -nodo.jugador)
        nodo.expandido = True
        # --- Evaluacion con la tabla de valores/heuristica, en vez de rollout ---
        v = valor_estimado(nodo.tablero)
        nodo.N += 1
        nodo.W += v
        return v

    # --- Seleccion: bajamos por el hijo con mejor puntuacion UCT ---
    mejor_m = max(nodo.hijos, key=lambda m: ucb_score(nodo, nodo.hijos[m]))
    v = simular(nodo.hijos[mejor_m])
    # --- Retropropagacion: el resultado sube y actualiza este nodo tambien ---
    nodo.N += 1
    nodo.W += v
    return v


### Lanzar muchas simulaciones desde una posición

`busqueda_mcts` crea la raíz para el tablero y jugador dados, llama a `simular` `num_simulaciones` veces, y devuelve cuántas visitas recibió cada movimiento posible desde ahí. Más visitas significa que el árbol de búsqueda lo considera más prometedor.


In [ ]:
def busqueda_mcts(tablero, jugador, num_simulaciones=60):
    """Devuelve {movimiento: visitas} tras num_simulaciones simulaciones."""
    raiz = Nodo(tablero, jugador)
    for _ in range(num_simulaciones):
        simular(raiz)
    return {m: hijo.N for m, hijo in raiz.hijos.items()}


**Primera demo: ejecuta la búsqueda desde un tablero vacío.** Con 60 simulaciones y X moviendo primero, fíjate en qué casilla acumula más visitas: si la búsqueda funciona bien, debería ser la casilla 4, el centro, porque participa en cuatro líneas ganadoras distintas.


In [ ]:
visitas_demo = busqueda_mcts(tablero_nuevo(), 1, num_simulaciones=60)
print("Visitas por movimiento desde el tablero vacio:", visitas_demo)


### De las visitas a una jugada: la mejora de política

`elegir_movimiento` convierte las visitas de la búsqueda en una política y elige un movimiento. Con temperatura > 0 hay exploración (útil en autojuego, para generar partidas variadas); con temperatura = 0 se juega siempre el movimiento más visitado (el modo que se usa al evaluar). Es la versión práctica de π′(a|s) ∝ Q(s,a)^(1/τ), la fórmula de mejora de política que viste hace un momento.


In [ ]:
def elegir_movimiento(visitas, temperatura):
    movimientos = list(visitas.keys())
    conteos = np.array([visitas[m] for m in movimientos], dtype=float)

    if temperatura == 0:
        return movimientos[int(np.argmax(conteos))]

    probabilidades = conteos ** (1.0 / temperatura)
    probabilidades /= probabilidades.sum()
    return np.random.choice(movimientos, p=probabilidades)


### Una partida completa de autojuego

`jugar_partida_autojuego` juega X contra O, ambos controlados por la misma búsqueda MCTS: es el autojuego propiamente dicho. Guarda cada tablero visitado, listo para actualizar la tabla de valores en cuanto se conozca el resultado final. El parámetro `mostrar`, que no existe en el mini-AlphaZero "de producción", solo sirve para proyectar la partida en clase.


In [ ]:
def jugar_partida_autojuego(num_simulaciones=60, temperatura=0.8, mostrar=False):
    """Juega una partida completa de X contra O, ambos controlados por la
    misma busqueda MCTS (autojuego). Devuelve los tableros visitados y el
    resultado final, listos para actualizar la tabla de valores."""
    tablero = tablero_nuevo()
    jugador = 1
    historial = []
    turno = 1

    while resultado(tablero) is None:
        visitas = busqueda_mcts(tablero, jugador, num_simulaciones)
        movimiento = elegir_movimiento(visitas, temperatura)
        historial.append(tablero)
        tablero = aplicar_movimiento(tablero, movimiento, jugador)
        if mostrar:
            print(f"Turno {turno}: juega {'X' if jugador == 1 else 'O'} en la casilla {movimiento}")
            imprimir_tablero(tablero)
        jugador = -jugador
        turno += 1

    return historial, resultado(tablero)


**Segunda demo — proyecta esta partida en clase.** La tabla de valores todavía está vacía, así que el agente juega solo con la heurística fija: es, más o menos, el nivel de una AlphaZero recién inicializada, antes de la primera generación de autojuego. *Guion de clase: pausa después de cada turno y pregunta si la jugada te parece razonable.*


In [ ]:
_, resultado_demo = jugar_partida_autojuego(mostrar=True)
print("Resultado:", {1: "Gana X", -1: "Gana O", 0: "Empate"}[resultado_demo])


### El paso de mejora: actualizar la tabla de valores

`actualizar_tabla_valores` acerca el valor estimado de cada posición visitada durante la partida hacia el resultado real, con una actualización tipo Monte Carlo / TD. Con el tiempo, la tabla deja de depender de la heurística inicial y refleja lo que el propio autojuego fue aprendiendo. Esto es, a pequeñísima escala, el entrenamiento de la red de valor de AlphaZero.


In [ ]:
def actualizar_tabla_valores(historial, resultado_final, alpha=0.5):
    for tablero in historial:
        actual = valor_estimado(tablero)
        tabla_valores[tablero] = actual + alpha * (resultado_final - actual)


### Cómo medimos el progreso: partidas contra un jugador aleatorio

Para saber si el agente mejora de verdad, lo enfrentamos a un rival que mueve al azar. `movimiento_aleatorio` es ese rival; `jugar_vs_aleatorio` juega una partida de evaluación con el agente MCTS (temperatura 0, es decir, jugando siempre su mejor movimiento según la tabla de valores actual) siempre como X.


In [ ]:
def movimiento_aleatorio(tablero):
    return random.choice(movimientos_validos(tablero))


def jugar_vs_aleatorio(num_simulaciones=60):
    tablero = tablero_nuevo()
    jugador = 1
    while resultado(tablero) is None:
        if jugador == 1:
            visitas = busqueda_mcts(tablero, jugador, num_simulaciones)
            movimiento = elegir_movimiento(visitas, temperatura=0)
        else:
            movimiento = movimiento_aleatorio(tablero)
        tablero = aplicar_movimiento(tablero, movimiento, jugador)
        jugador = -jugador
    return resultado(tablero)


### El bucle completo: autojuego + mejora, generación tras generación

Aquí se cierra el círculo. En cada generación: primero, 40 partidas de autojuego que alimentan la tabla de valores; después, 50 partidas de evaluación contra el jugador aleatorio, para medir cuánto ha mejorado el agente. Volvemos a fijar la semilla justo antes de empezar, para que el experimento sea reproducible sin importar qué celdas de demostración hayas ejecutado antes.

**Ejecuta la celda y observa el marcador de cada generación: es el mismo ciclo de autojuego que entrenó a AlphaGo, AlphaZero y AlphaStar, solo que aquí tarda segundos en vez de horas.**


In [ ]:
np.random.seed(0)
random.seed(0)

GENERACIONES = 6
PARTIDAS_AUTOJUEGO_POR_GENERACION = 40
PARTIDAS_EVALUACION = 50

for generacion in range(1, GENERACIONES + 1):
    # 1) Autojuego: generar partidas y usarlas para mejorar la tabla de valores
    for _ in range(PARTIDAS_AUTOJUEGO_POR_GENERACION):
        historial, resultado_final = jugar_partida_autojuego()
        actualizar_tabla_valores(historial, resultado_final)

    # 2) Evaluacion: medir el nivel actual contra un jugador aleatorio
    victorias, empates, derrotas = 0, 0, 0
    for _ in range(PARTIDAS_EVALUACION):
        r = jugar_vs_aleatorio()
        if r == 1:
            victorias += 1
        elif r == 0:
            empates += 1
        else:
            derrotas += 1

    print(
        f"Generacion {generacion}: "
        f"{victorias} victorias, {empates} empates, {derrotas} derrotas "
        f"(sobre {PARTIDAS_EVALUACION} partidas contra un jugador aleatorio) "
        f"| tabla de valores con {len(tabla_valores)} posiciones aprendidas"
    )


### Lectura del resultado

*Guion de clase: para aquí y pregunta a la clase qué esperaban ver en la primera generación frente a la última.*

En esta ejecución de referencia (con la semilla fijada en 0), el marcador es: generación 1, 50 victorias y 0 derrotas; generación 2, 49 victorias, 1 empate y 0 derrotas; generación 3, 48 victorias, 2 empates y 0 derrotas; generación 4, 50 victorias y 0 derrotas; generación 5, 48 victorias, 2 empates y 0 derrotas; generación 6, 47 victorias, 3 empates y 0 derrotas. El agente ya no pierde ni una sola partida desde la primera generación, y el número de empates y victorias va variando ligeramente según cuánto exploró cada tanda de autojuego. La tabla de valores, mientras tanto, crece de forma constante: pasa de 196 posiciones aprendidas tras la generación 1 a 748 tras la generación 6.

El Tres en Raya es tan pequeño que el patrón converge casi de inmediato, algo que ni de lejos ocurre con el ajedrez o el Go, donde AlphaZero necesitó horas de cómputo masivo —con hardware especializado— en vez de segundos en un portátil. La idea de fondo, sin embargo, es idéntica: autojuego que genera datos, una búsqueda que mejora la política local, y una actualización que traslada esa mejora de vuelta al conocimiento acumulado.


**Tercera demo — proyecta esta última partida en clase.** El agente ya entrenó seis generaciones: compárala con la primera demo, antes de entrenar, y pide a la clase que señale la diferencia.


In [ ]:
_, resultado_final_demo = jugar_partida_autojuego(temperatura=0.3, mostrar=True)
print("Resultado:", {1: "Gana X", -1: "Gana O", 0: "Empate"}[resultado_final_demo])


## Puntos clave

- Antes de AlphaGo, la IA ya jugaba a nivel sobrehumano en damas, ajedrez, backgammon y Scrabble, combinando búsqueda (minimax, poda alfa-beta) con funciones de valor, a veces aprendidas y a veces diseñadas a mano
- El MCTS es el puente entre esa historia y AlphaGo: busca mediante simulaciones en vez de resolver el juego completo, y UCT decide qué rama explorar con la misma lógica que UCB en los bandidos
- AlphaGo combinó una red de política, una red de valor y MCTS mediante PUCT, entrenadas primero por imitación de partidas humanas y después por autojuego, y venció a Lee Sedol cuatro partidas a una
- AlphaZero eliminó los datos humanos por completo: una sola red, un solo algoritmo, aplicado sin cambios a ajedrez, shōgi y Go, superando a Stockfish, Elmo y a la propia AlphaGo en pocas horas de autojuego, guiado por GPI y la fórmula π′(a|s) ∝ Q(s,a)^(1/τ)
- AlphaStar demostró que el mismo patrón —imitación más autojuego, esta vez organizado en una liga de agentes y ajustado con PPO— también funciona bajo información parcial y espacios de acción enormes, venciendo diez a cero a una profesional de StarCraft II
- El mini-AlphaZero de esta clase reproduce el mismo engranaje —autojuego, búsqueda tipo MCTS, mejora de política— a escala de Tres en Raya: mismo patrón, mismas piezas, sin necesitar más que `numpy`
- Cada sistema que has visto hoy es una recombinación de ideas que ya conocías: funciones de valor, políticas aprendidas por gradiente, control off-policy, aproximación con redes neuronales, y búsqueda tipo MCTS guiada por esas redes


## Antes de cerrar

**Abre el notebook de práctica de esta clase y completa los dos ejercicios.** Vas a programar tú misma o tú mismo, desde cero, el paso de selección y expansión de MCTS, y vas a adaptar este mismo esqueleto —autojuego, búsqueda, tabla de valores— a una versión reducida de Conecta 4.

## Y con esto, cierra el curso

Empezaste este curso con el dilema más simple posible de un solo bandido de dos brazos, decidiendo si explorar o explotar. Terminas viendo cómo esa misma tensión —explorar frente a explotar, evaluar frente a mejorar, actuar frente a planificar— escaló hasta vencer al mejor jugador de Go del mundo, reinventar el ajedrez sin ayuda humana y ganar diez a cero contra profesionales de un videojuego de tiempo real. No cambió la idea de fondo en ningún momento del camino: cambió la escala, el cómputo y la paciencia para dejar que el autojuego hiciera su trabajo.

Lo que has recorrido en estas 12 clases —bandidos, PDM, programación dinámica, Monte Carlo y TD, SARSA y Q-learning, DQN, gradientes de política, planificación, arquitecturas de agentes distribuidas, y ahora estos sistemas que hicieron historia— es una base sólida para leer los artículos originales de DeepMind, experimentar con librerías como Gymnasium o PettingZoo, o simplemente seguir jugando con tus propias variantes del código de esta clase. El campo sigue moviéndose rápido, y la mejor forma de seguirle el ritmo es la misma que aprendiste aquí: prueba, observa el resultado, ajusta la política y vuelve a intentarlo.

**Enhorabuena por llegar hasta el final del curso.** Ahora te toca a ti seguir explorando.
